# 8단계: 날씨 weak correction 도입

## 이 단계의 핵심 변화
날씨를 direct feature가 아니라 weak correction으로 바꾼 단계입니다.

## 왜 점수가 좋아졌나
날씨는 행동을 흔드는 보조 신호라 약한 보정으로 넣을 때 크게 개선됐습니다.

## 안내
이 파일은 다운로드 오류를 피하기 위해 새 이름으로 다시 만든 버전입니다.
코드 셀 안에는 기존 주석이 남아 있거나, 원본 설명 흐름을 유지합니다.


# 현재 최고 구조 + 날씨 weak correction 실험 (경로 자동탐색 수정본)

목표:
- 현재 최고 구조인 **참여율 기반 + 중식/석식 피처 분리 + 메뉴 weak correction** 유지
- 날씨는 모델 입력 피처로 강하게 넣지 않고,
  **최종 예측 참여율을 아주 작게 보정하는 weak correction 신호**로만 추가

출력 파일:
- `submission_weather_weak_correction.csv`

In [ ]:
# 필요시 한 번만 설치
# !pip install xgboost scikit-learn

In [ ]:
import os
import re
import random
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
from xgboost import XGBRegressor

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# 경로 자동 탐색
train_candidates = [
    r"C:\Users\yuzhd\github\team\data\made\train_median.csv",
    r"C:\Users\yuzhd\github\team\data\train_median.csv",
    r"/mnt/c/Users/yuzhd/github/team/data/made/train_median.csv",
    r"/mnt/c/Users/yuzhd/github/team/data/train_median.csv",
    r"./train_median.csv",
    r"./data/made/train_median.csv",
]

test_candidates = [
    r"C:\Users\yuzhd\github\team\data\test.csv",
    r"C:\Users\yuzhd\github\team\data\made\test.csv",
    r"/mnt/c/Users/yuzhd/github/team/data/test.csv",
    r"/mnt/c/Users/yuzhd/github/team/data/made/test.csv",
    r"./test.csv",
    r"./data/test.csv",
]

sample_sub_candidates = [
    r"C:\Users\yuzhd\github\team\data\sample_submission.csv",
    r"C:\Users\yuzhd\github\team\data\made\sample_submission.csv",
    r"/mnt/c/Users/yuzhd/github/team/data/sample_submission.csv",
    r"/mnt/c/Users/yuzhd/github/team/data/made/sample_submission.csv",
    r"./sample_submission.csv",
    r"./data/sample_submission.csv",
]

weather_candidates = [
    r"C:\Users\yuzhd\github\team\날씨\weather.csv",
    r"C:\Users\yuzhd\github\team\날씨\weather_no_snowfall.csv",
    r"/mnt/c/Users/yuzhd/github/team/날씨/weather.csv",
    r"/mnt/c/Users/yuzhd/github/team/날씨/weather_no_snowfall.csv",
    r"./weather.csv",
    r"./weather_no_snowfall.csv",
]

def find_existing_path(candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    return None

train_path = find_existing_path(train_candidates)
test_path = find_existing_path(test_candidates)
sample_sub_path = find_existing_path(sample_sub_candidates)
weather_path = find_existing_path(weather_candidates)

print("train_path:", train_path)
print("test_path:", test_path)
print("sample_sub_path:", sample_sub_path)
print("weather_path:", weather_path)

if train_path is None:
    raise FileNotFoundError("train_median.csv를 찾지 못했습니다. 후보 경로를 확인하세요.")
if test_path is None:
    raise FileNotFoundError("test.csv를 찾지 못했습니다. 후보 경로를 확인하세요.")
if sample_sub_path is None:
    raise FileNotFoundError("sample_submission.csv를 찾지 못했습니다. 후보 경로를 확인하세요.")
if weather_path is None:
    raise FileNotFoundError("weather.csv 또는 weather_no_snowfall.csv를 찾지 못했습니다. 후보 경로를 확인하세요.")

In [ ]:
train = pd.read_csv(train_path, encoding="utf-8-sig")
test = pd.read_csv(test_path, encoding="utf-8-sig")
sample_submission = pd.read_csv(sample_sub_path, encoding="utf-8-sig")
weather = pd.read_csv(weather_path, encoding="utf-8-sig")

train["일자"] = pd.to_datetime(train["일자"])
test["일자"] = pd.to_datetime(test["일자"])

display(train.head())
display(test.head())
display(weather.head())

In [ ]:
def find_col(cols, candidates):
    for cand in candidates:
        for c in cols:
            if cand.lower() == c.lower():
                return c
        for c in cols:
            if cand.lower() in c.lower():
                return c
    return None

date_col = find_col(weather.columns, ["일시", "date", "날짜"])
temp_col = find_col(weather.columns, ["기온", "평균기온", "temperature", "avg_temp"])
rain_col = find_col(weather.columns, ["강수량", "rainfall", "precipitation", "rain"])

weather = weather[[date_col, temp_col, rain_col]].copy()
weather = weather.rename(columns={date_col: "일자", temp_col: "기온", rain_col: "강수량"})
weather["일자"] = pd.to_datetime(weather["일자"])
weather["기온"] = pd.to_numeric(weather["기온"], errors="coerce")
weather["강수량"] = pd.to_numeric(weather["강수량"], errors="coerce")

weather["강수량"] = weather["강수량"].fillna(0)
weather["기온"] = weather["기온"].interpolate().bfill().ffill()
weather["is_rain"] = (weather["강수량"] > 0).astype(int)
weather["is_hot"] = (weather["기온"] >= 28).astype(int)
weather["is_cold"] = (weather["기온"] <= 5).astype(int)

train = train.merge(weather, on="일자", how="left")
test = test.merge(weather, on="일자", how="left")

for df_ in [train, test]:
    for c in ["기온", "강수량", "is_rain", "is_hot", "is_cold"]:
        if c in ["is_rain", "is_hot", "is_cold"]:
            df_[c] = df_[c].fillna(0)
        else:
            df_[c] = df_[c].interpolate().bfill().ffill()

display(train[["일자","기온","강수량","is_rain","is_hot","is_cold"]].head())

In [ ]:
def normalize_menu_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.replace("\n", " ").replace("\r", " ").replace("/", " ").replace("&", " ").replace("+", " ")
    text = text.replace("(", " ").replace(")", " ")
    text = re.sub(r"[^가-힣A-Za-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def add_features(df):
    df = df.copy()
    df["월"] = df["일자"].dt.month
    df["일"] = df["일자"].dt.day
    df["요일"] = df["일자"].dt.weekday
    df["식사가능자수"] = (
        df["본사정원수"] - df["본사휴가자수"] - df["본사출장자수"] - df["현본사소속재택근무자수"]
    ).clip(lower=1)

    month_end = df["일자"] + pd.offsets.MonthEnd(0)
    df["days_to_month_end"] = (month_end - df["일자"]).dt.days
    df["is_month_end_part"] = (df["days_to_month_end"] <= 5).astype(int)
    df["is_wed"] = (df["요일"] == 2).astype(int)
    df["is_fri"] = (df["요일"] == 4).astype(int)
    df["has_overtime"] = (df["본사시간외근무명령서승인건수"] > 0).astype(int)
    df["log_overtime"] = np.log1p(df["본사시간외근무명령서승인건수"])
    return df

train = add_features(train)
test = add_features(test)

train["중식참여율"] = (train["중식계"] / train["식사가능자수"]).clip(lower=0, upper=1.5)
train["석식참여율"] = (train["석식계"] / train["식사가능자수"]).clip(lower=0, upper=1.5)

lunch_features = [
    "월","일","요일","식사가능자수","본사출장자수","본사시간외근무명령서승인건수",
    "is_fri","days_to_month_end","is_month_end_part",
]
dinner_features = [
    "월","일","요일","식사가능자수","본사출장자수","본사시간외근무명령서승인건수",
    "is_wed","has_overtime","log_overtime",
]

In [ ]:
lunch_keyword_map = {
    "chicken":"치킨",
    "donkatsu":"돈까스",
    "jeyuk":"제육",
    "curry":"카레",
    "fried_rice":"볶음밥",
    "bibimbap":"비빔밥",
    "jjajang":"짜장",
    "tangsuyuk":"탕수육",
}

def lunch_keyword_flags(text):
    text = normalize_menu_text(text)
    return {f"kw_{k}": int(v in text) for k, v in lunch_keyword_map.items()}

def dinner_style_flags(text):
    text = normalize_menu_text(text)
    flags = {
        "style_korean":0, "style_chinese":0, "style_western":0,
        "style_japanese":0, "style_snack":0, "style_fusion":0,
    }
    chinese_kw = ["짜장","짬뽕","탕수","마파","깐풍","유산슬"]
    western_kw = ["돈까스","스테이크","파스타","스프","햄버거","피자","리조또"]
    japanese_kw = ["초밥","우동","가라아게","돈부리","소바"]
    snack_kw = ["떡볶이","순대","튀김","라볶이","김밥","분식","어묵"]
    fusion_kw = ["브리또","타코","샐러드","또띠아","퓨전"]

    if any(k in text for k in chinese_kw): flags["style_chinese"] = 1
    if any(k in text for k in western_kw): flags["style_western"] = 1
    if any(k in text for k in japanese_kw): flags["style_japanese"] = 1
    if any(k in text for k in snack_kw): flags["style_snack"] = 1
    if any(k in text for k in fusion_kw): flags["style_fusion"] = 1
    if sum(flags.values()) == 0: flags["style_korean"] = 1
    return flags

train_lunch_menu = pd.DataFrame([lunch_keyword_flags(x) for x in train["중식메뉴"].fillna("")])
test_lunch_menu = pd.DataFrame([lunch_keyword_flags(x) for x in test["중식메뉴"].fillna("")])

train_dinner_menu = pd.DataFrame([dinner_style_flags(x) for x in train["석식메뉴"].fillna("")])
test_dinner_menu = pd.DataFrame([dinner_style_flags(x) for x in test["석식메뉴"].fillna("")])

In [ ]:
params = {
    "n_estimators": 1000,
    "learning_rate": 0.05,
    "max_depth": 5,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "min_child_weight": 1,
    "gamma": 0.0,
}

X_train_lunch = train[lunch_features]
X_test_lunch = test[lunch_features]
y_lunch = train["중식참여율"]

X_train_dinner = train[dinner_features]
X_test_dinner = test[dinner_features]
y_dinner = train["석식참여율"]

xgb_lunch = XGBRegressor(objective="reg:squarederror", random_state=42, **params)
xgb_dinner = XGBRegressor(objective="reg:squarederror", random_state=42, **params)

xgb_lunch.fit(X_train_lunch, y_lunch)
xgb_dinner.fit(X_train_dinner, y_dinner)

base_pred_lunch_ratio = np.clip(xgb_lunch.predict(X_test_lunch), 0, 1.5)
base_pred_dinner_ratio = np.clip(xgb_dinner.predict(X_test_dinner), 0, 1.5)

train_pred_lunch_ratio = np.clip(xgb_lunch.predict(X_train_lunch), 0, 1.5)
train_pred_dinner_ratio = np.clip(xgb_dinner.predict(X_train_dinner), 0, 1.5)

train["lunch_residual"] = train["중식참여율"] - train_pred_lunch_ratio
train["dinner_residual"] = train["석식참여율"] - train_pred_dinner_ratio

In [ ]:
LUNCH_SHRINK = 0.35
DINNER_SHRINK = 0.35
RAW_CLIP = 0.03
FINAL_CLIP = 0.04

lunch_corr = {}
for col in train_lunch_menu.columns:
    mask = train_lunch_menu[col] == 1
    lunch_corr[col] = train.loc[mask, "lunch_residual"].mean() if mask.sum() >= 5 else 0.0

dinner_corr = {}
for col in train_dinner_menu.columns:
    mask = train_dinner_menu[col] == 1
    dinner_corr[col] = train.loc[mask, "dinner_residual"].mean() if mask.sum() >= 5 else 0.0

for k in lunch_corr:
    lunch_corr[k] = float(np.clip(lunch_corr[k] * LUNCH_SHRINK, -RAW_CLIP, RAW_CLIP))
for k in dinner_corr:
    dinner_corr[k] = float(np.clip(dinner_corr[k] * DINNER_SHRINK, -RAW_CLIP, RAW_CLIP))

lunch_menu_adj = np.zeros(len(test))
for col in test_lunch_menu.columns:
    lunch_menu_adj += test_lunch_menu[col].values * lunch_corr.get(col, 0.0)

dinner_menu_adj = np.zeros(len(test))
for col in test_dinner_menu.columns:
    dinner_menu_adj += test_dinner_menu[col].values * dinner_corr.get(col, 0.0)

lunch_menu_adj = np.clip(lunch_menu_adj, -FINAL_CLIP, FINAL_CLIP)
dinner_menu_adj = np.clip(dinner_menu_adj, -FINAL_CLIP, FINAL_CLIP)

In [ ]:
# 날씨 weak correction: 아주 작은 보정만 적용
weather_lunch_signal = (
    0.010 * test["is_rain"].values
    - 0.006 * test["is_hot"].values
    + 0.004 * test["is_cold"].values
)

weather_dinner_signal = (
    0.004 * test["is_rain"].values
    + 0.003 * test["is_cold"].values
)

weather_lunch_signal = np.clip(weather_lunch_signal, -0.02, 0.02)
weather_dinner_signal = np.clip(weather_dinner_signal, -0.015, 0.015)

final_pred_lunch_ratio = np.clip(base_pred_lunch_ratio + lunch_menu_adj + weather_lunch_signal, 0, 1.5)
final_pred_dinner_ratio = np.clip(base_pred_dinner_ratio + dinner_menu_adj + weather_dinner_signal, 0, 1.5)

print("중식 날씨 보정 평균:", weather_lunch_signal.mean(), "min/max:", weather_lunch_signal.min(), weather_lunch_signal.max())
print("석식 날씨 보정 평균:", weather_dinner_signal.mean(), "min/max:", weather_dinner_signal.min(), weather_dinner_signal.max())

In [ ]:
pred_lunch_count = np.clip(final_pred_lunch_ratio * test["식사가능자수"].values, 0, None)
pred_dinner_count = np.clip(final_pred_dinner_ratio * test["식사가능자수"].values, 0, None)

print("중식 식수 예측 평균:", pred_lunch_count.mean())
print("석식 식수 예측 평균:", pred_dinner_count.mean())

In [ ]:
submission = sample_submission.copy()

if "중식계" in submission.columns and "석식계" in submission.columns:
    lunch_col = "중식계"
    dinner_col = "석식계"
else:
    numeric_like_cols = [c for c in submission.columns if c != submission.columns[0]]
    lunch_col, dinner_col = numeric_like_cols[:2]

submission[lunch_col] = pred_lunch_count
submission[dinner_col] = pred_dinner_count

save_name = "submission_weather_weak_correction.csv"
submission.to_csv(save_name, index=False, encoding="utf-8-sig")

print("저장 완료:", save_name)
display(submission.head())